####这是最终运行代码的地方

In [13]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
%matplotlib inline
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import definition_lqr as df_lqr
import definition_algorithm as df_alg
torch.set_default_dtype(torch.float64)
torch.manual_seed(42)

####初始的矩阵和一些数

In [ ]:
# Define problem parameters from project spec
H = [[0.5, 0.5], [0.0, 0.5]]      # Drift matrix for state
M = [[1.0, 1.0], [0.0, 1.0]]      # Control influence matrix
sigma = np.eye(2) * 0.5           # Diffusion coefficient (noise strength)
C = [[1.0, 0.1], [0.1, 1.0]]      # Running cost for state
D = [[1.0, 0.1], [0.1, 1.0]]
D = (np.array(D) * 0.1).tolist()  # Running cost for control (set to Identity)
R = [[1.0, 0.3], [0.3, 1.0]]
R = (np.array(R) * 10.0).tolist() # Terminal cost
T = 0.5         # Terminal time
N = 1000        # Number of time steps
tau = 0.1       # Entropy regularization parameter
gamma = 10.0    # Control noise variance
dt = T / N      # Time step size
time_grid = torch.linspace(0, T, N + 1)  # Time discretization grid

####练习1.1的生成

In [ ]:
lqr = df_lqr.StrictLQR(H, M, sigma, C, D, R, T, N)
x = torch.tensor([1.0, 1.0], dtype=torch.float64)
y = torch.tensor([2.0, 2.0], dtype=torch.float64)

print("Value function v(0, x):", lqr.value_function(0, x))
print("Optimal control a(0, x):", lqr.optimal_control(0, x))
print("Value function v(0, y):", lqr.value_function(0, y))
print("Optimal control a(0, y):", lqr.optimal_control(0, y))

# Batch evaluation examples
t_list = torch.tensor([0.0, 0.1], dtype=torch.float64)
x_list = torch.stack([x, y])
print("Batch value v(t, x):", lqr.value_function_batch(t_list, x_list))
print("Batch control a(t, x):", lqr.optimal_control_batch(t_list, x_list))

####练习1.2的生成

In [ ]:
lqr_mc = df_lqr.LQRMonteCarlo(H, M, sigma, C, D, R, T, N)

# Experiment 1: Fix samples, vary time steps
time_steps_list = [2**i for i in range(1, 12)]
lqr_mc.run_experiment_1(num_samples=10000, time_steps_list=time_steps_list)

# Experiment 2: Fix time step, vary number of samples
num_samples_list = [2 * 4**i for i in range(6)]  # 2, 8, 32, ..., 2048
lqr_mc.run_experiment_2(fixed_time_steps=10000, num_samples_list=num_samples_list)

####练习2的生成

In [ ]:
x0_list = [
    torch.tensor([2.0, 2.0], dtype=torch.float64),
    torch.tensor([2.0, -2.0], dtype=torch.float64),
    torch.tensor([-2.0, -2.0], dtype=torch.float64),
    torch.tensor([-2.0, 2.0], dtype=torch.float64),
]

In [ ]:
soft_lqr = df_lqr.SoftLQR(H, M, sigma, C, D, R, T, N, tau, gamma)

soft_lqr.plot_trajectories(x0_list)
soft_lqr.print_value_and_controls(x0_list)

####练习3的生成

In [19]:
# === Train critic using supervised regression from fixed policy (Algorithm 2) ===
model = df_alg.Critic()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [31]:
# === Train critic using supervised regression from fixed policy (Algorithm 2) ===
model = df_alg.Critic()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

steps = 2000
batch_size = 32
loss_log = []
best_loss = float('inf')
best_model = None

for step in range(steps):
    loss = 0.0
    for _ in range(batch_size):
        t_seq, x_seq, a_seq, f_seq, gT = df_alg.generate_trajectory()
        for n in range(N):
            v_pred = model(t_seq[n:n+1], x_seq[n:n+1])   # v(t_n, x_n)
            logp = torch.sum(a_seq[n:]**2, dim=1)        # Approximation of log π
            target = torch.sum(f_seq[n:] + tau * logp) * dt + gT
            loss += N * (v_pred - target) ** 2           # Sum-based loss scaled by N

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_log.append(loss.item())

    # Save best model
    if loss.item() < best_loss:
        best_loss = loss.item()
        best_model = model.state_dict()

    if step % 500 == 0:
        print(f"Step {step}, Loss = {loss.item():.4e}")

RuntimeError: output with shape [1] doesn't match the broadcast shape [0]